# Data Pipeline Refresh
This notebook is used to download the original as-is datasets used in this project.

In [ ]:
import os
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

DATA_DIR = "../data/original"

# create the data/original folder if it doesnt already exist
os.makedirs(DATA_DIR, exist_ok=True)

# print a message to actually show the path so it can be verified
print(f"📂 Saving data to: {os.path.abspath(DATA_DIR)}")

### Fetch **CelesTrak SATCAT.csv**
CelesTrak has a query limit on how many times you can 

In [ ]:
# simple function to fetch the celestrak satcat.csv file
# no scraping needed here 
def fetch_celestrak():
    """
    Updates the local copy of satcat.csv
    """
    print("--- Fetching CelesTrak (SATCAT) ---")
    url = "https://celestrak.org/pub/satcat.csv"

    # join the file paths
    save_path = os.path.join(DATA_DIR, "satcat.csv")

    try:
        # use requests to download the file, use stream=True for large files
        response = requests.get(url, stream=True)
        
        # triggers an error if the link is broken
        response.raise_for_status()
        
        # get the date the last time the file was updated
        last_modified = response.headers.get("Last-Modified")
        if last_modified:
            print(f"📅 Server Last Update: {last_modified}")

        # no error has been thrown were good to save it.
        with open(save_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=1024):
                f.write(chunk)

        # output save directory.
        print(f"✅ Success! SATCAT saved to: {save_path}")
    except Exception as e:
        # output the error message.
        print(f"❌ Error downloading CelesTrak: {e}")

### Fetch **UCS UCS-Satellite-Database 5-1-2023.csv**

In [ ]:
def fetch_ucs():
    print("\n--- Fetching UCS Satellite Database ---")
    # there is no direct download link for the file, the file path depends on the latest version
    # and changes every time the database is updated.
    # so we have to scrape the landing page to find the link to the excel file.
    landing_page = "https://www.ucsusa.org/resources/satellite-database"

    # we have to define these headers so the download can pretend to be a real browser/person
    # then we use soup to read every line of the html
    headers = {
        # AI gave me a realistic user agent to use for this request, it should help avoid being blocked by the server
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
        "Connection": "keep-alive",
        "Referer": "https://www.google.com/" 
    }

    try:
        print("    Scraping the landing page...")
        response = requests.get(landing_page, headers=headers)
        
        if response.status_code != 200:
            print(f"   ❌ Blocked! Status Code: {response.status_code}")
            return

        # feed the response text to BeautifulSoup to parse the HTML
        soup = BeautifulSoup(response.text, 'html.parser')
        
        target_link = None
        
        # find all of the a tags in the html file
        for link in soup.find_all('a', href=True):
            # strip and convert to lower
            text = link.text.strip().lower()

            # if the link is for the database excel file
            # then set target_link to the href value
            if text == "database":
                target_link = link['href']
                print(f"   🎯 Found the link: {target_link}")
                break
        
        if not target_link:
            print("   ❌ Still could not find the link.")
            return

        # target url was found, join the landing page with the target link to get the full url
        full_url = urljoin(landing_page, target_link)

        print("   ⬇️  Downloading the Excel file...")
        
        file_response = requests.get(full_url, headers=headers, stream=True)
        
        last_modified = file_response.headers.get("Last-Modified")
        if last_modified:
            print(f"   📅 Server Last Update: {last_modified}")
            
        filename_from_url = full_url.split("/")[-1]
        print(f"   🏷️ Remote Filename: {filename_from_url}")
        # ------------------------------------
        
        # save the excel file to data/original
        excel_path = os.path.join(DATA_DIR, "UCS_raw.xlsx")
        
        with open(excel_path, 'wb') as f:
            for chunk in file_response.iter_content(chunk_size=1024):
                f.write(chunk)

        print("   🔄 Converting to CSV...")

        # simple conversion from excel to csv using pandas
        df = pd.read_excel(excel_path)        
        csv_path = os.path.join(DATA_DIR, "UCS-Satellite-Database.csv")
        df.to_csv(csv_path, index=False)
        
        print(f"   ✅ Success! Saved to: {csv_path}")

    except Exception as e:
        print(f"   ❌ Error with UCS data: {e}")

### Execute Fetch

In [ ]:
fetch_ucs()
print()
fetch_celestrak()